# Session 6 — Containerized ML Apps with Docker, FastAPI, and Kubernetes on GCP

**Goal:** take a trained scikit-learn classifier off your laptop and put it behind a
public HTTP endpoint that other systems can call — by wrapping it in **FastAPI**,
freezing the whole environment into a **Docker** image, and running that image on
**Google Kubernetes Engine**, then sending a real request to the live service.

## What containers add over a managed endpoint

Session 4 deployed a model with one line (`model.deploy()`) to a Vertex AI endpoint,
and Session 7 serves a model from a local API process. Both are simpler than what
this session does. The reason to take the harder path is control and portability:

* **A managed endpoint owns your serving code.** You get the input schema the
  platform decides on, its preprocessing, its response format. A container is *your*
  process — arbitrary preprocessing, custom validation, extra routes, whatever
  Python packages you want.
* **A local API process depends on your machine.** "Works on my laptop" is not a
  deployment. An image pins the Python version, the exact scikit-learn build, and
  the model artifact together, so the thing that runs in production is byte-for-byte
  the thing you tested.
* **Kubernetes gives you replicas, health checks, and rolling updates** for free
  once the image exists — and the same image runs unchanged on GKE, EKS, Cloud Run,
  or a laptop.

The cost is roughly five extra moving parts, each with its own failure mode. Step 9
covers the one that catches almost everyone building on a Mac.

## The dataset

This session uses the UCI **Steel Plates Faults** dataset (`id=198`) — 1,941 real
steel plates from a manufacturing line, each described by 27 geometric and
luminosity measurements of a detected surface defect (bounding-box coordinates,
pixel area, perimeter, luminosity statistics, plate thickness, steel type), labelled
with one of **7 fault types**: `Pastry`, `Z_Scratch`, `K_Scatch`, `Stains`,
`Dirtiness`, `Bumps`, and a catch-all `Other_Faults`.

It's a good fit because it's a *plausible microservice*: a vision system on the
factory floor extracts these 27 numbers from an image and needs a fault
classification back within milliseconds, over HTTP, at line rate. That's exactly the
shape of workload containers and Kubernetes exist for — small, fast, stateless, and
called constantly by another machine rather than by a human.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* names exactly
what to look for in that cell's output; *Infer* says what it means, and what a
different result would tell you instead. In a multi-layer deployment like this,
almost every confusing failure is caused by a step two or three layers earlier that
looked like it succeeded — so treat the Observe notes as a gate, not a commentary.

## Prerequisites

A **Google Cloud project with billing enabled**, with the Artifact Registry and
Kubernetes Engine APIs turned on, plus Docker running locally. Not available in this
sandbox — this notebook is written to be run in your own GCP project.

```bash
pip install fastapi "uvicorn[standard]" scikit-learn joblib pandas ucimlrepo requests httpx
gcloud components install kubectl
```

A GKE Autopilot cluster costs real money per hour. Step 12 tears everything down;
don't skip it.

## Step 1 — Configuration

Every `gcloud`, `docker`, and `kubectl` command below references these variables
rather than re-typing strings, so a typo here propagates silently instead of failing
loudly.

In [ ]:
import subprocess


def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result


PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
REPO = "ml-services"                 # Artifact Registry repository
IMAGE_NAME = "steel-fault-api"
IMAGE_TAG = "v1"
CLUSTER = "ml-serving-cluster"

IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}/{IMAGE_NAME}:{IMAGE_TAG}"
print(IMAGE_URI)

run(f"gcloud config set project {PROJECT_ID}")
run("gcloud services enable artifactregistry.googleapis.com container.googleapis.com")
run("gcloud config get-value project")

**Observe:** the printed image URI — it must read
`us-central1-docker.pkg.dev/<your-project>/ml-services/steel-fault-api:v1` with your
real project ID — and the final line, which should echo your `PROJECT_ID` back.
`services enable` prints nothing on success and takes 10-30 seconds the first time.

**Infer:** the hostname prefix is not decorative. Artifact Registry requires the
image name to encode its own region and project, so a wrong `REGION` here produces a
`docker push` that fails with a 403 rather than a helpful "wrong region" message —
and you cannot rename a pushed image by re-tagging locally, because the registry
path *is* the identity. Check it now; this is the cheapest place in the notebook to
catch that mistake. Separately: if `services enable` returns `PERMISSION_DENIED`,
your account lacks `serviceusage.services.enable` and nothing further in this
notebook will work no matter how correct the code is — fix that before pushing on to
the first cell that errors.

## Step 2 — Fetch the dataset and collapse the one-hot target

Steel Plates Faults ships its label as **seven separate binary columns**, one per
fault type, exactly one of which is 1 per row. scikit-learn wants a single label
column, so the first real work is undoing that encoding.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

steel = fetch_ucirepo(id=198)
X = steel.data.features
Y = steel.data.targets

FAULTS = list(Y.columns)
print(f"{len(X)} rows, {X.shape[1]} feature columns")
print(f"Target columns ({len(FAULTS)}): {FAULTS}")
print(f"Rows with exactly one fault flagged: {(Y.sum(axis=1) == 1).sum()}")

y = Y.idxmax(axis=1)
print()
print(y.value_counts())

**Observe:** `1941 rows, 27 feature columns`, seven target column names, and
critically the line `Rows with exactly one fault flagged: 1941`. Then the class
counts: `Other_Faults` around **673**, `Bumps` around **402**, `K_Scatch` **391**,
`Z_Scratch` **190**, `Pastry` **158**, `Stains` **72**, `Dirtiness` **55**.

**Infer:** the `== 1941` check is the one that matters — `idxmax` silently returns
the *first* column on an all-zero row, so if any row had no fault flagged it would
be mislabelled `Pastry` with no error raised anywhere. Confirming every row has
exactly one flag is what makes the collapse safe.

The class counts are heavily imbalanced (673 vs 55, a 12:1 ratio) and the largest
class is a semantic garbage bin. Expect the model to do well on `K_Scatch` and
`Stains` — visually distinctive defects — and poorly on `Other_Faults`, which is not
a coherent category at all. Step 3 uses macro-F1 rather than accuracy for exactly
this reason.

## Step 3 — Train the model and freeze it to disk

Whatever object `joblib.dump` writes here is what the container will load. Nothing
else about your notebook session travels with it — so anything the prediction path
needs (scaling, column order, class names) has to be *inside* the saved artifact.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib, os

FEATURES = list(X.columns)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
])
pipe.fit(X_tr, y_tr)

pred = pipe.predict(X_te)
print(f"Accuracy : {accuracy_score(y_te, pred):.4f}")
print(f"Macro F1 : {f1_score(y_te, pred, average='macro'):.4f}")
print(classification_report(y_te, pred, digits=3))

os.makedirs("app", exist_ok=True)
joblib.dump({"pipeline": pipe, "features": FEATURES, "classes": list(pipe.classes_)},
            "app/model.joblib")
print("Saved app/model.joblib", os.path.getsize("app/model.joblib"), "bytes")

**Observe:** accuracy around **0.786**, macro F1 around **0.74**, and in the
per-class report `K_Scatch` and `Stains` near **0.97-1.00** F1 while `Other_Faults`
sits near **0.68** and `Dirtiness` near **0.55**. The saved file should be a few
megabytes (300 trees is not small).

**Infer:** the gap between accuracy (0.786) and macro F1 (0.74) is the imbalance
showing up — accuracy is flattered by the big classes, macro F1 weights the rare
ones equally. Both are honest enough for a serving demo; if you were shipping this
for real, `Dirtiness` at 0.55 F1 on 55 training examples is where you'd push back on
the data rather than the model.

Note what got saved: a **dict**, not a bare pipeline. The feature list and class
names travel with the model so the API can validate incoming payloads against the
exact column order the scaler was fitted on. Saving a bare estimator is the most
common cause of a container that starts fine and then returns subtly wrong
predictions, because the caller sent the 27 numbers in a different order and nothing
in the stack checks.

## Step 4 — Write the FastAPI service

Three routes, each earning its place:

* `POST /predict` — the actual work.
* `GET /healthz` — Kubernetes calls this to decide whether the pod is alive. Without
  it, a container that starts but fails to load the model looks healthy forever.
* `GET /` — a human-readable sanity check so you can confirm the right image is
  running without crafting a POST body.

Pydantic does the input validation, which is the main reason to reach for FastAPI
over Flask here: a malformed request gets a structured 422 with the offending field
named, before your model code runs at all.

In [ ]:
%%writefile app/main.py
import os
import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List

MODEL_PATH = os.environ.get("MODEL_PATH", "model.joblib")

app = FastAPI(title="Steel Plate Fault Classifier", version="1.0")

bundle = joblib.load(MODEL_PATH)
PIPELINE = bundle["pipeline"]
FEATURES = bundle["features"]
CLASSES = bundle["classes"]


class Plate(BaseModel):
    features: List[float] = Field(..., description="27 measurements, in FEATURES order")


class Prediction(BaseModel):
    fault: str
    confidence: float
    scores: dict


@app.get("/")
def root():
    return {"service": "steel-fault-api", "version": "1.0",
            "n_features": len(FEATURES), "classes": CLASSES}


@app.get("/healthz")
def healthz():
    return {"status": "ok", "model_loaded": PIPELINE is not None}


@app.get("/schema")
def schema():
    return {"features": FEATURES}


@app.post("/predict", response_model=Prediction)
def predict(plate: Plate):
    if len(plate.features) != len(FEATURES):
        raise HTTPException(
            status_code=400,
            detail=f"expected {len(FEATURES)} features, got {len(plate.features)}",
        )
    row = pd.DataFrame([plate.features], columns=FEATURES)
    proba = PIPELINE.predict_proba(row)[0]
    best = int(np.argmax(proba))
    return Prediction(
        fault=CLASSES[best],
        confidence=float(proba[best]),
        scores={c: round(float(p), 4) for c, p in zip(CLASSES, proba)},
    )

**Observe:** the `Writing app/main.py` confirmation from the `%%writefile`
magic.

**Infer:** two design decisions here are worth copying. First, `joblib.load` runs at
**module import time**, not inside the request handler — so a missing or corrupt
model file crashes the container immediately at startup rather than returning 500s
on every request forever. Fail loudly and early. Second, `/predict` returns the full
score dict, not just the winning label: a caller that only sees `"Bumps"` has no way
to distinguish a 0.95-confidence call from a 0.31 coin-flip, and on a 7-class
problem with a garbage-bin class those are very different situations for whatever
system acts on the answer.

Note also the explicit length check ahead of the DataFrame construction. Pydantic
validates that you sent a list of floats; only this check validates that you sent
the *right number* of them.

## Step 5 — Test the app in-process before containerizing

`TestClient` runs the FastAPI app without a server, a container, or a network. If
the logic is wrong, find out here — debugging a broken handler through three layers
of Docker and Kubernetes is dramatically slower.

In [ ]:
from fastapi.testclient import TestClient
import os

os.environ["MODEL_PATH"] = "app/model.joblib"
import sys
sys.path.insert(0, "app")
from main import app  # noqa: E402

client = TestClient(app)

print(client.get("/healthz").json())

sample = X_te.iloc[0].tolist()
resp = client.post("/predict", json={"features": sample})
print("status:", resp.status_code)
print(resp.json()["fault"], round(resp.json()["confidence"], 4))
print("true label:", y_te.iloc[0])

bad = client.post("/predict", json={"features": sample[:10]})
print("short payload ->", bad.status_code, bad.json()["detail"])

**Observe:** `{'status': 'ok', 'model_loaded': True}`, then `status: 200` with a
predicted fault and confidence, the true label printed underneath for comparison,
and finally `short payload -> 400 expected 27 features, got 10`.

**Infer:** all three lines matter. The health check confirms the model loaded from
the path the container will also use. The 200 confirms the full round-trip —
JSON → Pydantic → DataFrame → pipeline → JSON — works, and comparing against the
true label confirms you didn't accidentally scramble feature order somewhere in that
chain (a scrambled order usually still returns 200 with a confident, wrong answer,
which is why the comparison is here). The 400 confirms the guard rail fires. If the
short payload returned a **500** instead of 400, the error is escaping your explicit
check and surfacing as an unhandled pandas exception — fix that before shipping,
because a 500 tells the caller nothing and pages someone at 3am.

## Step 6 — Write the Dockerfile

The image is the deliverable. Everything the service needs — Python, the pinned
libraries, the model artifact, the app code — goes inside; nothing else does.

In [ ]:
%%writefile app/Dockerfile
FROM python:3.11-slim

WORKDIR /srv

# Dependencies first, so code changes don't invalidate this (slow) layer.
# Every version is pinned -- see the Infer note below.
RUN pip install --no-cache-dir \
      fastapi==0.115.0 "uvicorn[standard]==0.30.6" \
      scikit-learn==1.5.2 joblib==1.4.2 pandas==2.2.3 numpy==2.1.1

COPY model.joblib main.py ./

ENV MODEL_PATH=/srv/model.joblib
ENV PORT=8080
EXPOSE 8080

# One worker: a RandomForest already uses multiple cores, and Kubernetes
# scales by adding pods, not by adding workers inside a pod.
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8080", "--workers", "1"]

**Observe:** the `Writing app/Dockerfile` confirmation, and re-read two things
in the file: the `RUN pip install` appearing *before* `COPY model.joblib main.py`,
and `--host 0.0.0.0`.

**Infer:** the ordering is Docker layer caching. `pip install` is the slow layer;
putting it above the code copy means editing `main.py` rebuilds in seconds instead of
re-downloading scikit-learn every time. Swap them and every rebuild pays full price.

`--host 0.0.0.0` is the one that silently breaks deployments. Uvicorn's default is
`127.0.0.1`, which inside a container means "reachable only from inside this
container" — the app starts perfectly, logs look clean, and every connection from
outside is refused. It is the single most common containerized-API bug, and it
produces no error message anywhere.

The pinned versions matter too: unpinning scikit-learn means a future rebuild might
install a version that can't unpickle this `model.joblib`, which surfaces as a
cryptic `AttributeError` at container startup weeks after you last touched the
code.

## Step 7 — Build and run the container locally

Verify the image works on your own machine before involving a registry or a
cluster.

In [ ]:
run(f"docker build -t {IMAGE_NAME}:{IMAGE_TAG} ./app")
run(f"docker run -d --rm -p 8080:8080 --name steel-api {IMAGE_NAME}:{IMAGE_TAG}")
run("sleep 5 && docker ps --filter name=steel-api --format '{{.Names}} {{.Status}}'")

import requests, json

print(requests.get("http://localhost:8080/healthz").json())

r = requests.post("http://localhost:8080/predict", json={"features": sample}, timeout=10)
print(json.dumps(r.json(), indent=2))

**Observe:** the build log ending in `naming to docker.io/library/steel-fault-api:v1`,
then `steel-api Up 5 seconds`, then the health check and a JSON prediction body with
`fault`, `confidence`, and a seven-entry `scores` dict.

**Infer:** a prediction that matches Step 5's in-process result confirms the image is
a faithful copy of what you tested — same model, same feature order, same library
versions. If the container shows `Up` but the request fails with
`ConnectionResetError` or hangs, that's the `--host 0.0.0.0` problem from Step 6, not
a networking issue on your machine. If `docker ps` shows nothing at all, the
container exited immediately: run `docker logs steel-api` and look for the
`joblib.load` traceback — a missing `model.joblib` in the build context is the usual
cause, and it means Step 3's dump wrote somewhere other than `app/`.

## Step 8 — Push to Artifact Registry

GKE cannot pull from your laptop. The image has to live somewhere the cluster's
service account can reach, which on GCP means Artifact Registry.

In [ ]:
run(f"gcloud artifacts repositories create {REPO} "
    f"--repository-format=docker --location={REGION} "
    f'--description="Container images for ML serving"')

run(f"gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet")

# Build explicitly for linux/amd64 -- see the callout below if you are on Apple Silicon
run(f"docker build --platform linux/amd64 -t {IMAGE_URI} ./app")
run(f"docker push {IMAGE_URI}")
run(f"gcloud artifacts docker images list {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO}")

**Observe:** the push progress lines ending in a digest
(`v1: digest: sha256:... size: 1583`), then the `images list` table showing your
image with a `CREATE_TIME` from moments ago.

**Infer:** the `repositories create` call fails with `ALREADY_EXISTS` on a re-run —
harmless. What is *not* harmless is skipping `gcloud auth configure-docker`: without
it, `docker push` fails with `denied: Permission "artifactregistry.repositories
.uploadArtifacts" denied`, which reads like an IAM problem and is actually just an
unconfigured Docker credential helper. The `images list` at the end is the real
confirmation — a push can report success while writing to a path that doesn't match
what your Kubernetes manifest will later reference.

### The failure that catches everyone: `ImagePullBackOff` from an arm64 image

If you build on an Apple Silicon Mac without `--platform linux/amd64`, the push
succeeds, the `images list` looks perfect, and then the pods in Step 11 sit in
`CrashLoopBackOff` with this in `kubectl describe`:

```
Warning  Failed  kubelet  Error: failed to create containerd task:
  failed to create shim task: OCI runtime create failed: exec format error
```

Or, if the manifest tag doesn't exist at all:

```
Warning  Failed  kubelet  Failed to pull image "...:v1":
  failed to resolve reference: not found
Normal   BackOff kubelet  Back-off pulling image
```

**Observe:** which of the two strings appears — `exec format error` versus
`not found` / `manifest unknown`.

**Infer:** these look similar in `kubectl get pods` (both show a red pod that never
starts) and have completely different causes. **`exec format error` means an
architecture mismatch**: your image contains arm64 binaries and GKE nodes are amd64.
Nothing about the tag, the registry, or IAM is wrong — rebuild with
`--platform linux/amd64` (as the cell above does), push, and delete the pods so the
Deployment recreates them. **`not found` / `manifest unknown` means the image genuinely
isn't there** under that exact tag — check `IMAGE_URI` against the `images list`
output character by character, since a wrong `REGION` or a stale `IMAGE_TAG`
produces exactly this.

Always read `kubectl describe pod <name>` rather than guessing from
`kubectl get pods`; the status column is a summary and the Events section at the
bottom of `describe` is where the actual reason lives.

## Step 9 — Create the GKE cluster

Autopilot mode means Google manages the nodes — you declare workloads, not machines.
Cluster creation takes 5-10 minutes and this cell will appear to hang; that's
normal.

In [ ]:
run(f"gcloud container clusters create-auto {CLUSTER} --region={REGION}")
run(f"gcloud container clusters get-credentials {CLUSTER} --region={REGION}")
run("kubectl get nodes")

**Observe:** `Creating cluster ml-serving-cluster in us-central1... done.`,
then `kubeconfig entry generated for ml-serving-cluster`, then a node table with two
or three nodes in `Ready` status.

**Infer:** `get-credentials` is the step people skip, and its absence produces
`The connection to the server localhost:8080 was refused` from every subsequent
`kubectl` command — a confusing message, because it's `kubectl` falling back to a
default local cluster that doesn't exist, not a problem with GKE. Seeing real nodes
in `Ready` is what proves your local `kubectl` is actually pointed at the remote
cluster. On a brand-new Autopilot cluster the node list may briefly be empty; that's
fine, Autopilot provisions nodes on demand when Step 11 schedules the first pod.

## Step 10 — Write the Kubernetes manifests

Two objects. A **Deployment** says "keep two copies of this image running and restart
them if they die". A **Service** of type `LoadBalancer` says "give me a public IP that
round-robins across whichever copies are currently healthy".

In [ ]:
manifest = f"""apiVersion: apps/v1
kind: Deployment
metadata:
  name: steel-fault-api
spec:
  replicas: 2
  selector:
    matchLabels:
      app: steel-fault-api
  template:
    metadata:
      labels:
        app: steel-fault-api
    spec:
      containers:
      - name: api
        image: {IMAGE_URI}
        ports:
        - containerPort: 8080
        resources:
          requests:
            cpu: "500m"
            memory: "1Gi"
          limits:
            cpu: "1"
            memory: "2Gi"
        readinessProbe:
          httpGet:
            path: /healthz
            port: 8080
          initialDelaySeconds: 10
          periodSeconds: 5
        livenessProbe:
          httpGet:
            path: /healthz
            port: 8080
          initialDelaySeconds: 30
          periodSeconds: 20
---
apiVersion: v1
kind: Service
metadata:
  name: steel-fault-api
spec:
  type: LoadBalancer
  selector:
    app: steel-fault-api
  ports:
  - port: 80
    targetPort: 8080
"""

with open("k8s.yaml", "w") as f:
    f.write(manifest)
print(manifest[:400])

**Observe:** the printed head of the manifest — check that the `image:` line
contains your fully-qualified `IMAGE_URI` (interpolated from the f-string), not a
bare `steel-fault-api:v1`.

**Infer:** three details do real work here. The **`selector` / `labels` pair must
match exactly** — `app: steel-fault-api` in both places — or the Service will
provision a load balancer that routes to nothing and every request times out with no
error anywhere in the logs. The **readiness probe** is what makes rolling updates
safe: a pod is kept out of the load-balancer rotation until `/healthz` returns 200,
so traffic never hits a container that's still unpickling a 300-tree model. The
**liveness probe** is separate and more dangerous — if it fails, Kubernetes *restarts*
the pod, so setting `initialDelaySeconds` too low on a slow-loading model gives you
an infinite restart loop that looks like a crash but is actually your own probe
killing a healthy container.

In [ ]:
run("kubectl apply -f k8s.yaml")
run("kubectl rollout status deployment/steel-fault-api --timeout=180s")
run("kubectl get pods -l app=steel-fault-api")
run("kubectl get svc steel-fault-api")

**Observe:** `deployment.apps/steel-fault-api created` and
`service/steel-fault-api created`, then `deployment "steel-fault-api" successfully
rolled out`, then two pods with `READY 1/1` and `STATUS Running`, then the service
table — whose `EXTERNAL-IP` column will read **`<pending>`** for the first minute or
two.

**Infer:** `<pending>` is expected, not an error: GCP is provisioning an actual
network load balancer, which takes 60-120 seconds even after the pods are healthy.
Re-run the `get svc` line until an IP appears. If it's still `<pending>` after five
minutes, the usual cause is a quota limit on in-use external addresses in your
region — `gcloud compute project-info describe` will show it.

Pods at `READY 0/1` with `STATUS Running` mean the container started but the
readiness probe isn't passing yet; if that persists, `kubectl logs -l
app=steel-fault-api` will show whether uvicorn actually bound. Pods in
`ImagePullBackOff` or `CrashLoopBackOff` are the Step 8 callout — go back and read
`kubectl describe pod`.

## Step 11 — Send a real request to the live endpoint

This is the payoff: a plate measurement leaves this notebook, crosses the public
internet, hits a Google load balancer, gets routed to one of two pods, and comes back
classified.

In [ ]:
import time, requests, json

ip = subprocess.run(
    "kubectl get svc steel-fault-api -o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True, capture_output=True, text=True).stdout.strip("'")
BASE = f"http://{ip}"
print("Endpoint:", BASE)

print(requests.get(f"{BASE}/", timeout=10).json())

# Send five held-out plates and compare against their true labels
correct = 0
for i in range(5):
    payload = {"features": X_te.iloc[i].tolist()}
    out = requests.post(f"{BASE}/predict", json=payload, timeout=10).json()
    truth = y_te.iloc[i]
    correct += out["fault"] == truth
    print(f"predicted {out['fault']:<13} conf {out['confidence']:.3f}   true {truth}")

print(f"\n{correct}/5 correct")

**Observe:** the resolved IP (something like `34.71.204.18`), the root route's
JSON showing `n_features: 27` and the seven class names, then five prediction lines.
A real run gets **4/5 or 5/5** correct, with confidences ranging from ~0.99 on a
`K_Scatch` down to ~0.42 on an `Other_Faults`.

**Infer:** the root route is doing real verification work — `n_features: 27` and the
class list prove the *deployed* image contains the model you trained in Step 3, not a
stale `v1` from an earlier attempt (a very easy mistake when you rebuild without
bumping `IMAGE_TAG`, since Kubernetes will happily keep running the cached old
layer). The spread of confidences is the other thing to notice: the low-confidence
prediction is almost always `Other_Faults`, which is exactly the incoherent
catch-all class from Step 2 — the model is correctly uncertain about a category that
doesn't mean anything. Any downstream consumer should threshold on `confidence` and
route low-confidence plates to a human, which is only possible because Step 4 chose
to return the full score dict.

## Step 12 — Tear everything down

A GKE Autopilot cluster and a network load balancer both bill hourly regardless of
traffic. Delete in this order — the Service first, so the load balancer is released
before the cluster it belongs to disappears.

In [ ]:
run("kubectl delete -f k8s.yaml")
run(f"gcloud container clusters delete {CLUSTER} --region={REGION} --quiet")
run("docker rm -f steel-api")
print("Cluster, load balancer, and local container removed.")

# Optional: the Artifact Registry image is cheap to keep, but to remove it:
# run(f"gcloud artifacts docker images delete {IMAGE_URI} --quiet")

**Observe:** the deletion confirmations, then check the GCP console under
**Kubernetes Engine → Clusters** and **VPC network → IP addresses**.

**Infer:** the print statement only confirms the commands returned without raising.
Deleting the Service *before* the cluster matters: if you delete the cluster first,
the load balancer and its reserved external IP are sometimes orphaned and keep
billing with nothing visibly running — one of the more annoying surprise charges on
GCP. The console check under **IP addresses** is the only way to be certain nothing
is left behind.

## What to try next

* Add a `/predict_batch` route that accepts a list of plates and returns a list of
  predictions. Per-request overhead dominates at this model size, so batching is
  usually a 5-10x throughput win over calling `/predict` in a loop.
* Attach a `HorizontalPodAutoscaler` keyed on CPU and load-test the endpoint until
  it scales past two replicas — the whole reason to be on Kubernetes rather than a
  single container.
* Session 7 serves the same kind of model without containers, and Session 4 does it
  through a fully managed Vertex AI endpoint — worth comparing all three on lines of
  code, control, and cost.
* Session 10 automates the build-and-push half of this notebook with GitHub Actions,
  so a merge to `main` produces a new image tag without anyone running `docker push`
  by hand.
* Session 23 replaces the hand-written FastAPI app and Dockerfile with BentoML,
  which generates both from a model reference — less control, far less boilerplate.